In [1]:
import pandas as pd
from docling.document_converter import DocumentConverter
import json
from pathlib import Path
from typing import List
from pprint import pprint
import re

/Volumes/data_storage/python_project/docling_WBG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
summary_folderpath = r"extracted_manual/summary_table"
questions_folderpath = r"extracted_manual/questionaires"

In [3]:
def list_filepaths(folderpath: str) -> List[str]:
    folder = Path(folderpath)
    filepaths = [str(p) for p in folder.glob("*") if p.is_file()]
    return filepaths

In [4]:
def load_data_from_pdf(source: str):
    converter = DocumentConverter()
    result = converter.convert(source)
    
    return result.document

In [5]:
questions_filepaths = list_filepaths(folderpath=questions_folderpath)

In [6]:
questions_filepaths    

['extracted_manual/questionaires/B-READY-MH-2026-Q-Labor.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-Financial-Service.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-Business-Location.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-Taxation.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-Business-Entry.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-International-Trade.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-Dispute-Resolution.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-Utility.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-Market-Competition.pdf',
 'extracted_manual/questionaires/B-READY-MH-2026-Q-Business-Insolvency.pdf']

In [7]:
summary_filepaths = list_filepaths(folderpath=summary_folderpath)

In [8]:
summary_filepaths

['extracted_manual/summary_table/B-READY-MH-2026-Market-Competition.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Financial-service.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-utility-service.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Business-location.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-International-trade.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-labor.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Business-Insolvency.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Taxation.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Dispute-Resolution.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-business-entry.pdf']

In [9]:
doc = load_data_from_pdf(source=questions_filepaths[0])

In [10]:
doc.export_to_dict().keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [11]:
doc_dict = doc.export_to_dict()

In [12]:
def extract_questions(doc_dict: dict) -> List[str]:
    questions = []
    for item in doc_dict["texts"]:
        if item["label"] == "section_header" or item["label"] == "list_item":
            questions.append(item["orig"])
    
    return questions

In [13]:
questions = extract_questions(doc_dict=doc_dict)  

In [14]:
questions

['LABOR QUESTIONNAIRE',
 "1.1 WORKERS' CONDITIONS",
 '1.1.1 Labor Rights',
 '1. Does  the  legal  framework  provide  all  workers  with  the  right  of  freedom  of  association  and assembly? (Y/N)',
 '2. Does the legal framework provide all workers with the right to collective bargaining? (Y/N)',
 '3. Does the legal framework provide for all terms and conditions of employment (e.g., wages, working time, paid leave and benefits, occupational safety and health, grievance and dismissal/termination procedures) to be a subject of Collective Bargaining? (Y/N)',
 '4. Does the legal framework explicitly prohibit forced labor? (Y/N)',
 '5. What is the minimum age for admission to employment (non-hazardous, standard employment) established by law? [numerical value]',
 '6. What  is  the  minimum  age  established  by  law,  below  which  individuals  are  prohibited  from performing hazardous work? [ numerical value]',
 '7. Does the legal framework explicitly provide equal remuneration for wor

In [15]:
def questions_to_excel(questions: list[str]) -> None:
    topic = questions[0]
    sub_section_header_pattern = r'^\d+\.\d+\.\d+$'
    section_header_pattern = r'^\d+\.\d+$'
    
    rows = []
   
    for i in range(1, len(questions)):
        row = {}
        if bool(re.match(section_header_pattern, questions[i].split(" ")[0])):
            row["section_header"] = questions[i]
            row["sub_section_header"] = None
            row["question"] = None
        elif bool(re.match(sub_section_header_pattern, questions[i].split(" ")[0])):
            row["section_header"] = None
            row["sub_section_header"] = questions[i]
            row["question"] = None
        else:
            row["section_header"] = None
            row["sub_section_header"] = None
            row["question"] = questions[i]
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_excel(f"./questions_extracted/{topic}-question.xlsx")
    
    return

In [16]:
questions_to_excel(questions=questions)

In [17]:
for path in questions_filepaths:
    doc = load_data_from_pdf(source=path)
    doc_dict = doc.export_to_dict()
    questions = extract_questions(doc_dict=doc_dict)
    questions_to_excel(questions=questions)

In [18]:
summary_filepaths

['extracted_manual/summary_table/B-READY-MH-2026-Market-Competition.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Financial-service.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-utility-service.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Business-location.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-International-trade.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-labor.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Business-Insolvency.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Taxation.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-Dispute-Resolution.pdf',
 'extracted_manual/summary_table/B-READY-MH-2026-business-entry.pdf']

In [ ]:
for path in summary_filepaths:
    summary_doc = load_data_from_pdf(source=path)
    summary_doc_dict = summary_doc.export_to_dict()
    pprint(summary_doc_dict["tables"])
    

[{'annotations': [],
  'captions': [{'$ref': '#/texts/1'}],
  'children': [{'$ref': '#/texts/1'}],
  'content_layer': 'body',
  'data': {'grid': [[{'bbox': {'b': 205.28208752647276,
                               'coord_origin': 'TOPLEFT',
                               'l': 84.65694879600001,
                               'r': 374.5061951841,
                               't': 198.37824421180005},
                      'col_span': 2,
                      'column_header': False,
                      'end_col_offset_idx': 2,
                      'end_row_offset_idx': 1,
                      'fillable': False,
                      'row_header': False,
                      'row_section': False,
                      'row_span': 1,
                      'start_col_offset_idx': 0,
                      'start_row_offset_idx': 0,
                      'text': 'Pillar I-Quality of Regulations that Promote '
                              'Market Competition (96 indicators)'},
         